In [ ]:
!python --version

In [ ]:
!pip install -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as plt
import scipy as sp
from sklearn.metrics import rand_score
%matplotlib inline

## EDA


In [ ]:
df = pd.read_csv("spiral-dataset.csv", sep="\t", header=None, names=["x", "y", "class"])
df.head()

In [ ]:
df["class"].unique()

## 1.2.3.Generate a figure from the given dataset that resembles Figure 1.


In [ ]:
df.plot.scatter(x="x", y="y", c="class", colormap='viridis')

### Implement the k-means clustering algorithm. And do the following:

### • 2.a) Run your k-means algorithm on the given dataset setting the value k=3 (because visually we only have 3 clusters to worry about). And do not forget to randomly initialize the 3 centroids.


In [ ]:
# function to generate 3 random centroids within the bounds of the data
def random_centroids():
    # find the bounds of the data (doesn't need to be normalized per the instructions)
    max_x = df["x"].max()
    max_y = df["y"].max()

    # generate 3 random centroids within the bounds of the data
    centroids = [
        (np.random.rand() * max_x, np.random.rand() * max_y),
        (np.random.rand() * max_x, np.random.rand() * max_y),
        (np.random.rand() * max_x, np.random.rand() * max_y)
    ]

    # convert to a numpy array and return
    return np.array(centroids)



# function to find the closest centroid for each data point
def find_classes(df, centroids):
    # this is going to be an array of the closest centroid for each data point
    closest_centroids = []

    # build the array of closest centroids
    for index, row in df.iterrows():
        # get an array of the distances from the data point to each centroid
        distances = np.array([sp.spatial.distance.euclidean((row["x"], row["y"]), centroid) for centroid in centroids])
        # find the index of the closest centroid and add it to the list
        best_centroid = np.argmin(distances)
        closest_centroids.append(best_centroid)

    return np.array(closest_centroids)

Now we're going to make a function to iterate:

1. Attach a kmeans_cluster value to each row
2. Use the new dataframe with the kmeans_cluster column to find the mean point for the cluster.
3. Move the centroid to the mean point of each cluster.
4. Repeat until it converges


In [ ]:
def do_kmeans(input_df, debug=False):

    # make a copy of the input dataframe to work with
    kmeans_df = input_df.copy()

    # initialize the centroids and assign the initial clusters
    centroids = random_centroids()
    if debug:
        print("Initial centroids:")
        print(f"\t({centroids[0][0]:.2f}, {centroids[0][1]:.2f})")
        print(f"\t({centroids[1][0]:.2f}, {centroids[1][1]:.2f})")
        print(f"\t({centroids[2][0]:.2f}, {centroids[2][1]:.2f})")

    clusters = find_classes(kmeans_df, centroids)

    # initialize some variables to prepare the preconditions for the loop
    old_centroids = [
        (0,0),
        (0,0),
        (0,0)
    ]
    kmeans_df["kmeans_cluster"] = clusters

    # ITERATION: loop until the centroids don't move
    while not np.array_equal(centroids, old_centroids):

        # split the dataframe into three clusters based on the kmeans_cluster column
        cluster1 = kmeans_df[kmeans_df["kmeans_cluster"] == 0]
        cluster2 = kmeans_df[kmeans_df["kmeans_cluster"] == 1]
        cluster3 = kmeans_df[kmeans_df["kmeans_cluster"] == 2]

        # store the old centroids before updating them
        old_centroids = centroids.copy()

        # update the centroids to be the mean of the points in each cluster
        # if a cluster is empty, don't move the centroid
        centroids[0] = (cluster1["x"].mean(), cluster1["y"].mean()) if len(cluster1) > 0 else centroids[0]
        centroids[1] = (cluster2["x"].mean(), cluster2["y"].mean()) if len(cluster2) > 0 else centroids[1]
        centroids[2] = (cluster3["x"].mean(), cluster3["y"].mean()) if len(cluster3) > 0 else centroids[2]

        # print the centroid movement
        if debug:
            if not np.array_equal(centroids[0], old_centroids[0]):
                print(f"Centroid 1 moved from ({old_centroids[0][0]:.2f}, {old_centroids[0][1]:.2f}) to ({centroids[0][0]:.2f}, {centroids[0][1]:.2f})")
            if not np.array_equal(centroids[1], old_centroids[1]):
                print(f"Centroid 2 moved from ({old_centroids[1][0]:.2f}, {old_centroids[1][1]:.2f}) to ({centroids[1][0]:.2f}, {centroids[1][1]:.2f})")
            if not np.array_equal(centroids[2], old_centroids[2]):
                print(f"Centroid 3 moved from ({old_centroids[2][0]:.2f}, {old_centroids[2][1]:.2f}) to ({centroids[2][0]:.2f}, {centroids[2][1]:.2f})")

        # reassign the clusters based on the new centroids
        kmeans_df["kmeans_cluster"] = find_classes(kmeans_df, centroids)
    # END ITERATION
    
    if debug:
        print("Final centroids:")
        print(f"\t({centroids[0][0]:.2f}, {centroids[0][1]:.2f})")
        print(f"\t({centroids[1][0]:.2f}, {centroids[1][1]:.2f})")
        print(f"\t({centroids[2][0]:.2f}, {centroids[2][1]:.2f})")

    # we need to add 1 to the kmeans_cluster column to match the original class labels
    kmeans_df["kmeans_cluster"] = kmeans_df["kmeans_cluster"] + 1

    # return the dataframe with the kmeans_cluster column and the final centroids to be used for the SSE calculation
    return kmeans_df, centroids

In [ ]:
kmeans_df, centroids = do_kmeans(df, debug=True)

## • 2.b) Once your k-means algorithm has converged above, stop and from your clustering result compute the intrinsic performance metric: Sum of Squared Error, SSE (smaller the better), and the extrinsic performance metric: Rand-Index, RI (higher the better).


Let's find SSE first


In [ ]:
def kmeans_sse(df, centroids):
    # our running total
    total = 0

    # iterate through each row
    for index, row in df.iterrows():
        # select the centroid for the cluster that the row belongs to based on the column we added to the dataframe
        centroid = centroids[int(row["kmeans_cluster"] - 1)]
        # the error is the distance from the data point to the centroid
        error = sp.spatial.distance.euclidean((row["x"], row["y"]), centroid)
        # add the squared error to the total
        total += error ** 2

    return total

In [ ]:
sse = kmeans_sse(kmeans_df, centroids)
print(f"SSE: {sse}")
ri = rand_score(kmeans_df["class"], kmeans_df["kmeans_cluster"])
print(f"Rand Index: {ri}")

## • 2.c) Repeat Task (2.a) & (2.b) another 9 (nine) times randomizing again the initial centroids,and report out of the 10 runs of k-means what is the best SSE & RI you could get.


In [ ]:
# some lists to hold our results, initialize with the results from the first attempt
kmeans_df_list = [kmeans_df]
centroids_list = [centroids]
sse_list = [sse]
ri_list = [ri]

# re-print this because I have OCD
print(f"Attempt 01 -> SSE: {sse:.10f}, Rand Index: {ri:.10f}")

# loop 9 times
for i in range(9):
    # get the results for this attempt
    this_df, this_centroids = do_kmeans(df)
    this_sse = kmeans_sse(this_df, this_centroids)
    this_ri = rand_score(this_df["class"], this_df["kmeans_cluster"])

    # print the results for this attempt
    print(f"Attempt {i+2:02d} -> SSE: {this_sse:.10f}, Rand Index: {this_ri:.10f}")
    
    # append the results to the lists
    kmeans_df_list.append(this_df)
    centroids_list.append(this_centroids)
    sse_list.append(this_sse)
    ri_list.append(this_ri)

# find the best one
best_sse_i = np.argmin(sse_list)
best_ri_i = np.argmax(ri_list)
print(f"\nBest SSE:        Attempt {best_sse_i+1:02d} with SSE:     {sse_list[best_sse_i]:.10f}")
print(f"Best Rand Index: Attempt {best_ri_i+1:02d} with Rand Index: {ri_list[best_ri_i]:.10f}")

# sanity check
if best_sse_i != best_ri_i:
    print("What??? SSE and Rand Index don't agree on the best attempt?")

## • 2.d) Please draw the clustering results (like Figure 1).


In [ ]:
kmeans_df_list[best_sse_i].plot.scatter(x="x", y="y", c="kmeans_cluster", colormap='viridis', title=f"Best SSE model (Attempt {best_sse_i+1:02d})")

In [ ]:
kmeans_df_list[best_ri_i].plot.scatter(x="x", y="y", c="kmeans_cluster", colormap='viridis', title=f"Best RI model (Attempt {best_ri_i+1:02d})")

-
-
-
-
-
-
-
-


## (40 pts) Implement the Hierarchical clustering algorithm. And do the following:
## • 3.a) Using the “single linkage” method, run the hierarchical clustering algorithm on the dataset, and get a 3-cluster result (by cutting the dendrogram at a certain height), and report SSE and RI.

The cell below is my implementation of pairwise distance.  I have deactivated it in favor of scipy's `pdist()` function

In [7]:
import numpy as np
import pandas as pd
import matplotlib as plt
import scipy as sp
from sklearn.metrics import rand_score
%matplotlib inline

In [8]:
###########################################################################################################
####### I implemented pairwise distance myself, but I am ditching it in favor of scipy's pdist() ##########
###########################################################################################################
# build the distance matrix
# rows = []

# # iterate through each row
# for i, row in df[["x", "y"]].iterrows():
#     # get distance to every other point
#     row_distances = sp.spatial.distance.cdist(row.to_numpy().reshape(1, -1), df[["x", "y"]])
#     # add to matrix
#     rows.append(row_distances)

# # fix the dimensions
# distance_matrix = np.array([row.flatten() for row in rows])
# distance_matrix.shape



# reload the data
df = pd.read_csv("spiral-dataset.csv", sep="\t", header=None, names=["x", "y", "class"])

# pdist() is so much easier...
distance_matrix = sp.spatial.distance.pdist(df[["x", "y"]])
distance_matrix = sp.spatial.distance.squareform(distance_matrix)

# where we'll save the class values
single_linkage_classes = np.arange(0, len(distance_matrix))

# EDA
print(f"Distance matrix shape: {distance_matrix.shape}")
print(distance_matrix[:5, :5])

Distance matrix shape: (312, 312)
[[0.         1.03077641 1.98494332 2.97741499 3.88104367]
 [1.03077641 0.         0.95524866 1.94743421 2.85043856]
 [1.98494332 0.95524866 0.         0.99247166 1.90065778]
 [2.97741499 1.94743421 0.99247166 0.         0.91787799]
 [3.88104367 2.85043856 1.90065778 0.91787799 0.        ]]


In [9]:
# we're going to ignore the diaganol and lower triangle since those are duplicate values. fill them in with infinity so they are ignored in argmin()
distance_matrix[np.tril_indices_from(distance_matrix)] = np.inf
print(distance_matrix[:5, :5])

[[       inf 1.03077641 1.98494332 2.97741499 3.88104367]
 [       inf        inf 0.95524866 1.94743421 2.85043856]
 [       inf        inf        inf 0.99247166 1.90065778]
 [       inf        inf        inf        inf 0.91787799]
 [       inf        inf        inf        inf        inf]]


In [10]:
while len(np.unique(single_linkage_classes)) > 3:
    print(f"Unique classes: {len(np.unique(single_linkage_classes))}")

    # find the closest two points
    closest_i = np.argmin(distance_matrix)
    closest_i = np.unravel_index(closest_i, distance_matrix.shape)

    # collapse the class values of the two closest points into the larger of the two clusters

    # create a mask to match all the current members of each cluster
    x_matches = (single_linkage_classes[closest_i[0]] == single_linkage_classes)
    y_matches = (single_linkage_classes[closest_i[1]] == single_linkage_classes)
    # count how many points are in each cluster
    num_x_matches = x_matches.sum()
    num_y_matches = y_matches.sum()
    
    # assign the smaller cluster to the larger cluster
    if num_x_matches > num_y_matches:
        single_linkage_classes[y_matches] = single_linkage_classes[closest_i[0]]
    elif num_y_matches > num_x_matches:
        single_linkage_classes[x_matches] = single_linkage_classes[closest_i[1]]
    else:
        single_linkage_classes[y_matches] = single_linkage_classes[closest_i[0]]

    # remove the points so they don't get classified again
    distance_matrix[closest_i[0], :] = np.inf
    distance_matrix[:, closest_i[1]] = np.inf

Unique classes: 312
Unique classes: 311
Unique classes: 310
Unique classes: 309
Unique classes: 308
Unique classes: 307
Unique classes: 306
Unique classes: 305
Unique classes: 304
Unique classes: 303
Unique classes: 302
Unique classes: 301
Unique classes: 300
Unique classes: 299
Unique classes: 298
Unique classes: 297
Unique classes: 296
Unique classes: 295
Unique classes: 294
Unique classes: 293
Unique classes: 292
Unique classes: 291
Unique classes: 290
Unique classes: 289
Unique classes: 288
Unique classes: 287
Unique classes: 286
Unique classes: 285
Unique classes: 284
Unique classes: 283
Unique classes: 282
Unique classes: 281
Unique classes: 280
Unique classes: 279
Unique classes: 278
Unique classes: 277
Unique classes: 276
Unique classes: 275
Unique classes: 274
Unique classes: 273
Unique classes: 272
Unique classes: 271
Unique classes: 270
Unique classes: 269
Unique classes: 268
Unique classes: 267
Unique classes: 266
Unique classes: 265
Unique classes: 264
Unique classes: 263


In [5]:
single_linkage_classes

array([  2,   2,   4,   4,   7,   7,   7,   9,   9,  11,  11,  12,  14,
        14,  15,  18,  18,  18,  19,  21,  21,  22,  26,  26,  26,  26,
        27,  29,  29,  31,  31,  33,  33,  36,  36,  36,  38,  38,  39,
        43,  43,  43,  43,  44,  45,  48,  48,  48,  49,  51,  51,  54,
        54,  54,  57,  57,  57,  59,  59,  60,  62,  62,  64,  64,  67,
        67,  67,  68,  70,  70,  72,  72,  74,  74,  77,  77,  77,  80,
        80,  80,  82,  82,  84,  84,  86,  86,  87,  90,  90,  90,  94,
        94,  94,  94,  96,  96,  98,  98, 101, 101, 101, 102, 104, 104,
       105, 105, 107, 109, 109, 111, 111, 113, 113, 114, 116, 116, 117,
       120, 120, 120, 122, 122, 123, 125, 125, 128, 128, 128, 129, 131,
       131, 133, 133, 135, 135, 137, 137, 141, 141, 141, 141, 145, 145,
       145, 145, 147, 147, 150, 150, 150, 151, 153, 153, 155, 155, 157,
       157, 159, 159, 161, 161, 163, 163, 165, 165, 168, 168, 168, 170,
       170, 173, 173, 173, 176, 176, 176, 177, 180, 180, 180, 18

• 3.b) Using the “complete linkage” method, run the hierarchical clustering algorithm on the
dataset, and get a 3-cluster result (by cutting the dendrogram at a certain height), and report
SSE and RI.
• 3.c) Using the “average linkage” method, run the hierarchical clustering algorithm on the
dataset, and get a 3-cluster result (by cutting the dendrogram at a certain height), and report
SSE and RI.
• 3.d) Using the “centroid linkage” method, run the hierarchical clustering algorithm on the
dataset, and get a 3-cluster result (by cutting the dendrogram at a certain height), and report
SSE and RI.
• 3.e) Please comment, out of the 4 clustering results (3.a), (3.b), (3.c) and (3.d) which
method gets you the best SSE as well as RI.
• 3.f) Please draw the clustering results (like Figure 1).
